##### Copyright 2026 Google LLC.

In [2]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: JSON Mode Quickstart

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/JSON_mode.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The Gemini API can be used to generate a JSON output if you set the schema that you would like to use.

Two methods are available. You can either set the desired output in the prompt or supply a schema to the model separately.

> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/JSON_mode.ipynb).

### Install dependencies

In [ ]:
%pip install -U -q "google-genai>=2.9.0"  # 2.0 for Interactions API

### Configure your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for a walkthrough.

In [9]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

## Set your constrained output in the prompt

For this first example just describe the schema you want in the prompt itself, and ask the model to return JSON using `response_format`.

The `response_format` parameter with `{"type": "text", "mime_type": "application/json"}` tells the model to output **only valid JSON** — without it, the model might wrap the JSON in markdown code blocks or add explanatory text around it.

In [11]:
prompt = """
    List a few popular cookie recipes using this JSON schema:

    Recipe = {'recipe_name': str}
    Return: list[Recipe]
"""

Select the model you want to use in this guide:

In [13]:
MODEL_ID = "gemini-3.6-flash" # @param ["gemini-3.5-flash-lite", "gemini-3.6-flash", "gemini-2.5-flash", "gemini-3-flash-preview", "gemini-2.5-flash-preview", "gemini-2.5-pro", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

raw_interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    response_format={"type": "text", "mime_type": "application/json"},
)

Parse the string to JSON:
> The response from `interactions.create` contains a list of `steps`. For text responses, access the output via `interaction.steps[-1].content[0].text`.


In [15]:
import json

response = json.loads(raw_interaction.steps[-1].content[0].text)
print(response)

[{'recipe_name': 'Chocolate Chip Cookies'}, {'recipe_name': 'Oatmeal Raisin Cookies'}, {'recipe_name': 'Peanut Butter Cookies'}, {'recipe_name': 'Sugar Cookies'}, {'recipe_name': 'Snickerdoodles'}]


For readability serialize and print it:

In [17]:
print(json.dumps(response, indent=4))

[
    {
        "recipe_name": "Chocolate Chip Cookies"
    },
    {
        "recipe_name": "Oatmeal Raisin Cookies"
    },
    {
        "recipe_name": "Peanut Butter Cookies"
    },
    {
        "recipe_name": "Sugar Cookies"
    },
    {
        "recipe_name": "Snickerdoodles"
    }
]


## Supply the schema to the model directly

Another option is to pass a schema directly via the `response_format` parameter. The model output will then follow that schema exactly.

You can define your schema as a Python class using [Pydantic](https://docs.pydantic.dev/latest/) (or `typing_extensions.TypedDict`) and use `.model_json_schema()` to generate the JSON Schema automatically. This is much cleaner than writing raw JSON Schema by hand:

In [19]:
from pydantic import BaseModel, Field
from typing import List, Optional

class Ingredient(BaseModel):
    """A single ingredient in a recipe."""
    name: str = Field(description="Name of the ingredient.")
    quantity: str = Field(description="Quantity of the ingredient, including units.")

class Recipe(BaseModel):
    """A recipe with name, description, ingredients, and instructions."""
    recipe_name: str = Field(description="The name of the recipe.")
    recipe_description: str = Field(description="A one-sentence description of the recipe.")
    ingredients: List[Ingredient]

Use `.model_json_schema()` to automatically generate the JSON Schema from your Pydantic model. This is the recommended approach — it avoids writing long JSON Schema dictionaries by hand:

In [21]:
# Preview the generated schema
import json
print(json.dumps(Recipe.model_json_schema(), indent=2))

In [ ]:
result = client.interactions.create(
    model=MODEL_ID,
    input="List a few imaginative cookie recipes along with a one-sentence description as if you were a gourmet restaurant and their main ingredients",
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": Recipe.model_json_schema(),
    },
)

# Parse and validate the response directly into a Pydantic object
recipes = json.loads(result.steps[-1].content[0].text)
print(json.dumps(recipes, indent=4))

You can also validate the response directly into your Pydantic model using `model_validate_json()` for type safety:

In [ ]:
# Validate into Pydantic objects
from pydantic import TypeAdapter

recipes_list = TypeAdapter(List[Recipe]).validate_json(result.steps[-1].content[0].text)
for r in recipes_list:
    print(f"🍪 {r.recipe_name}: {r.recipe_description}")
    for ing in r.ingredients:
        print(f"   - {ing.quantity} {ing.name}")
    print()

In [22]:
# You can also use TypedDict if you prefer not to depend on Pydantic:
import typing_extensions as typing

class SimpleRecipe(typing.TypedDict):
    recipe_name: str
    recipe_description: str

[
    {
        "recipe_name": "Lavender & Wildflower Honey Shortbread",
        "recipe_description": "A delicate, buttery shortbread infused with hand-harvested culinary lavender and finished with a glaze of artisanal wildflower honey.",
        "recipe_ingredients": [
            "European-style butter",
            "all-purpose flour",
            "granulated sugar",
            "dried lavender buds",
            "wildflower honey",
            "Maldon sea salt"
        ]
    },
    {
        "recipe_name": "Umami Miso & White Chocolate Chunk",
        "recipe_description": "A nuanced exploration of sweet and savory notes blending fermented white miso with velvety white chocolate and toasted macadamia nuts.",
        "recipe_ingredients": [
            "White miso paste",
            "white chocolate chunks",
            "toasted macadamia nuts",
            "brown butter",
            "flour",
            "cane sugar"
        ]
    },
    {
        "recipe_name": "Smoked Sea Salt 

> **Tip:** The Pydantic approach with `model_json_schema()` is the recommended method — it keeps your schema in sync with your code, provides built-in validation, and avoids manual JSON Schema errors.

## Next Steps
### Useful API references:

Check the [structured output](https://ai.google.dev/gemini-api/docs/interactions/structured-output) documentation for more details, including:
* **Streaming** structured outputs
* **Structured outputs with tools** (Google Search, Code Execution, etc.) — available with Gemini 3 models
* **JSON Schema support** — supported types and properties
* **Structured outputs vs function calling** — when to use which

### Related examples

* The constrained output is used in the [Text summarization](../examples/json_capabilities/Text_Summarization.ipynb) example to provide the model a format to summarize a story (genre, characters, etc...)
* The [Object detection](../examples/json_capabilities/Object_detection.ipynb) example shows how to detect the position of specific objects in an image
* Check all the [JSON](../examples/json_capabilities/) examples.